<a href="https://colab.research.google.com/github/SAO-P/Basic-Deep-Learning-2026-Spring/blob/main/lecture02_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第2回講義 演習

今回は，深層モデルやそのライブラリは用いず，機械学習の基礎を理解するために，まず回帰問題を扱い，その後ロジスティック回帰およびソフトマックス回帰を実装します．

なお，ロジスティック回帰やソフトマックス回帰は名前に「回帰」とありますが，分類問題であることに注意してください.  
シグモイド関数を用いることで，データのラベルが 1 または 0 である確率を出力します．

---

# Lecture 2: Exercise

This time, we will not use deep models or their libraries, but in order to understand the basics of machine learning, we will first deal with regression problems and then implement logistic regression and softmax regression.

Please note that although logistic regression and softmax regression have the word ``regression'' in their names, they are classification problems.
By using the sigmoid function, the probability that the data label is 1 or 0 is output.


## 目次

1. [【課題 1】回帰問題と平均二乗誤差 (MSE)](#scrollTo=t4lVHe-VtcDY&line=5&uniqifier=1)

    1.1. [データセットの作成]()

    1.2. [モデル・損失関数・学習方法]()

    1.3. [学習前の予測]()

    1.4. [学習]()

    1.5. [学習結果の確認]()

1. [【課題 2】ロジスティック回帰の実装と学習 (OR)](#scrollTo=quBPRwBf4Kq7&line=1&uniqifier=1)
    
    2.1. [シグモイド関数](#scrollTo=awSlFgRA4Kq9)

    2.2. [データセットの設定と重みの定義](#scrollTo=HAA-lvhz4KrF)
    
    2.3. [train関数とvalid関数](#scrollTo=7raMb3ts4KrL)
    
    2.4. [学習](#scrollTo=LiuO_6B-4Krb)

1. [【課題 3】ソフトマックス回帰の実装と学習 (MNIST)](#scrollTo=44tdPsW34Krq&line=1&uniqifier=1)
    
    3.1. [ソフトマックス関数](#scrollTo=YEprLDMd4Krr)
    
    3.2. [データセットの設定と重みの定義](#scrollTo=52fR-55x4Krx)
    
    3.3. [train関数とvalid関数](#scrollTo=80EOFI-n4Kr6)

    3.4. [学習](#scrollTo=JBGInXhG4KsJ)

---

## Table of Contents

1. [[Assignment 1] Regression problem and mean square error (MSE)](#scrollTo=t4lVHe-VtcDY&line=5&uniqifier=1)

    1.1. [Create dataset]()

    1.2. [Model/loss function/learning method]()

    1.3. [Prediction before learning]()

    1.4. [Learn]()

    1.5. [Check learning results]()

1. [[Assignment 2] Implementation and learning of logistic regression (OR)](#scrollTo=quBPRwBf4Kq7&line=1&uniqifier=1)
    
    2.1. [Sigmoid function](#scrollTo=awSlFgRA4Kq9)

    2.2. [Dataset settings and weight definition](#scrollTo=HAA-lvhz4KrF)
    
    2.3. [train function and valid function](#scrollTo=7raMb3ts4KrL)
    
    2.4. [Learning](#scrollTo=LiuO_6B-4Krb)

1. [[Assignment 3] Implementation and learning of softmax regression (MNIST)](#scrollTo=44tdPsW34Krq&line=1&uniqifier=1)
    
    3.1. [Softmax function](#scrollTo=YEprLDMd4Krr)
    
    3.2. [Dataset settings and weight definition](#scrollTo=52fR-55x4Krx)
    
    3.3. [train function and valid function](#scrollTo=80EOFI-n4Kr6)

    3.4. [Learning](#scrollTo=JBGInXhG4KsJ)


In [1]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from keras.datasets import mnist
import matplotlib.pyplot as plt

import numpy as np

np.random.seed(34)

## 1.【課題 1】回帰問題と平均二乗誤差 (MSE)

まず最も基本的な例として，連続値を予測する回帰問題を考える．  
ここでは，ニューラルネットワークの学習の基本構造を，  
平均二乗誤差 (Mean Squared Error; MSE)を用いて確認する．

---

## 1. [Assignment 1] Regression problem and mean square error (MSE)

First, as the most basic example, consider a regression problem that predicts continuous values.  
Here, we will explain the basic structure of neural network learning.
Confirm using mean squared error (MSE).


### 1.1. データセットの作成

1次元入力 $x$ に対して，連続値 $y$ を出力する回帰データを作成する．  
ここでは，ノイズを含む線形関数を用いる．

---

### 1.1. Creating a dataset

Create regression data that outputs continuous values $y$ for one-dimensional input $x$.  
Here, we use a linear function that includes noise.


In [ ]:
# Number of data
N = 100

# input
x = np.random.uniform(-1, 1, (N, 1))

# Ground truth value (linear + noise)
y = 2 * x + 1 + 0.1 * np.random.randn(N, 1)

# visualization
plt.scatter(x, y, alpha=0.6)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Regression dataset")
plt.show()

### 1.2. モデル・損失関数・学習方法

回帰問題では，入力 $x$ から出力 $\hat{y}$ を予測するモデルを学習する．  
ここでは最も単純なニューラルネットワークとして，線形変換を用いる．

$\hat{y} = Wx + b$

モデルの学習には，予測値と正解値の差を測る  
平均二乗誤差 (Mean Squared Error; MSE)を用いる．

$\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^N (y_i - \hat{y}_i)^2$

勾配降下法により，MSE を最小化するように  
パラメータ $W, b$ を更新する．

---

### 1.2. Model, loss function, learning method

In regression problems, we learn a model that predicts the output $\hat{y}$ from the input $x$.  
Here, we use linear transformation as the simplest neural network.

$\hat{y} = Wx + b$

To learn the model, measure the difference between the predicted value and the correct value.
Use the mean squared error (MSE).

$\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^N (y_i - \hat{y}_i)^2$

Gradient descent is used to minimize MSE.
Update parameters $W, b$.


In [ ]:
# Parameter initialization
W = np.random.randn(1, 1)
b = np.zeros((1,))

# mean square error
def mse(y, y_hat):
    return np.mean((y - y_hat) ** 2)

# learning function
def train(x, y, W, b, lr=0.1):
    # prediction
    y_hat = x @ W + b

    # loss
    loss = mse(y, y_hat)

    # Gradient calculation
    dW = 2 * np.mean((y_hat - y) * x)
    db = 2 * np.mean(y_hat - y)

    # Parameter update
    W -= lr * dW
    b -= lr * db

    return W, b, loss

### 1.3. 学習前の予測

初期化直後のパラメータ $W, b$ を用いた予測を可視化する．  
この時点では，モデルはまだデータの構造を学習していない．

---

### 1.3. Prediction before learning

Visualize the prediction using parameters $W, b$ immediately after initialization.  
At this point, the model has not yet learned the structure of the data.


In [ ]:
# Prediction before learning
y_pred_init = x @ W + b

# sort input x
idx = np.argsort(x[:, 0])
x_sorted = x[idx]
y_pred_init_sorted = y_pred_init[idx]

plt.scatter(x, y, alpha=0.6, label="True")
plt.plot(x_sorted, y_pred_init_sorted, color="gray", linestyle="--", label="Before training")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Before training")
plt.legend()
plt.show()

### 1.4. 学習

勾配降下法によりパラメータを更新し，MSE が減少することを確認する．

---

### 1.4. Learning

Update the parameters using gradient descent and confirm that the MSE decreases.


In [ ]:
epochs = 100
lr = 0.1
losses = []

for epoch in range(epochs):
    W, b, loss = train(x, y, W, b, lr=lr)
    losses.append(loss)

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Training loss (MSE)")
plt.show()

print("W:", W.ravel())
print("b:", b)

### 1.5. 学習結果の確認

学習後のモデルが，入力 $x$ に対してどのような予測 $\hat{y}$ を出すかを可視化する．

---

### 1.5. Confirm learning results

Visualize what kind of prediction $\hat{y}$ the trained model makes for the input $x$.


In [ ]:
# Prediction after learning
y_pred = x @ W + b

# Sort input x to make it easier to draw lines
idx = np.argsort(x[:, 0])
x_sorted = x[idx]
y_pred_sorted = y_pred[idx]

plt.scatter(x, y, alpha=0.6, label="True")
plt.plot(x_sorted, y_pred_sorted, color="red", label="After training")
plt.xlabel("x")
plt.ylabel("y")
plt.title("After training")
plt.legend()
plt.show()

## 2.【課題 2】ロジスティック回帰の実装と学習 (OR)

---

## 2. [Assignment 2] Implementation and learning of logistic regression (OR)


### 2.1. シグモイド関数（Sigmoid function）
$$
    \sigma({\bf x}) = \frac{1}{1 + \exp(-{\bf x})} = \frac{\exp({\bf x})}{1 + \exp({\bf x})}
$$


In [ ]:
def sigmoid(x):
    # simple implementation
    #  return 1 / (1 + np.exp(-x))


    # Implementation with exp overflow countermeasures
    # When x >=0, sigmoid(x) = 1 / (1 + exp(-x))
    # When x < 0, sigmoid(x) = exp(x) / (1 + exp(x))
    return  # WRITE ME


x = np.arange(-10, 10, 0.1)
plt.plot(x, sigmoid(x.copy()))

### 2.2. データセットの設定と重みの定義

次にデータセットと，パラメータの初期化を行います．

今回扱うORゲートは，二つの入力の少なくともどちらかが1であれば1を出力し，入力が両方とも0であれば0を出力します．

これが線形な分類器で実現可能な問題であることは，下図からもわかります．

---

### 2.2. Dataset settings and weight definitions

Next, initialize the dataset and parameters.

The OR gate we will be dealing with this time outputs 1 if at least one of its two inputs is 1, and outputs 0 if both inputs are 0.

You can see from the figure below that this problem is achievable with a linear classifier.


In [ ]:
# OR dataset
x_train_or = np.array([[0, 1], [1, 0], [0, 0], [1, 1]])  #  (4, 2)

y_train_or = np.array([[1], [1], [0], [1]])              #  (4, 1)

x_valid_or, y_valid_or = x_train_or, y_train_or
#  x_test_or, y_test_or = x_train_or, y_train_or


plt.figure(figsize=(6, 6))
plt.hlines([0], xmin=-1, xmax=2, color="black", alpha=0.7)
plt.vlines([0], ymin=-1, ymax=2, color="black", alpha=0.7)

plt.scatter(*x_train_or[y_train_or.squeeze() == 0].T, color="red", label="0")
plt.scatter(*x_train_or[y_train_or.squeeze() == 1].T, color="blue", label="1")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.xlim([-0.5, 1.5])
plt.ylim([-0.5, 1.5])
plt.legend()
plt.show()

次にパラメータを初期化します．重みは一様分布からのサンプリング，バイアスは0で初期化を行います．

---

Next, initialize the parameters. The weights are sampled from a uniform distribution and the bias is initialized to 0.


In [ ]:
# Weights (number of dimensions of input layer: 2, number of dimensions of output layer: 1)
W_or = np.random.uniform(low=-0.08, high=0.08, size=(2, 1)).astype('float32')
b_or = np.zeros(shape=(1,)).astype('float32')

### 2.3. train関数とvalid関数

---

### 2.3. train and valid functions


**目的関数（Objective function）**

2クラス交差エントロピー誤差関数（2-class cross entropy error function）
$$ E ({\bf x}, {\bf y}; {\bf W}, {\bf b} ) =  -\frac{1}{N}\sum^N_{i=1} \left[ {\bf y}_i \log {\bf \hat{y}}_i ({\bf x}_i; {\bf W}, {\bf b}) + (1 - {\bf y}_i) \log \{ 1 - {\bf \hat{y}}_i ({\bf x}_i; {\bf W}, {\bf b}) \}\right] $$


**モデルの推論（Model Inference）**
$$
    {\bf \hat{y}}_i = \sigma({\bf W}^T {\bf x}_i + {\bf b})
$$

**モデルの学習（Training the model）**

勾配降下法により学習します．クロスエントロピー誤差関数 $E$ のパラメータに関する勾配は以下のように簡単に計算できるので，それに基づいて更新を行っていきます．

（Learning is performed using gradient descent. The gradient regarding the parameters of the cross-entropy error function $E$ can be easily calculated as shown below, so we will update it based on it.）

\begin{align*}
    \delta_i &= {\bf \hat{y}}_i - {\bf y}_i \\
    \nabla_{\bf W} E &= \frac{1}{N}\sum^N_{i=1}\delta_i {\bf x}^{\mathrm{T}}_i \\
    \nabla_{\bf b} E &= \frac{1}{N}\sum^N_{i=1}\delta_i  \\
    {\bf W} &\leftarrow {\bf W} - \epsilon \nabla_{\bf W} E \\
    {\bf b} &\leftarrow {\bf b} - \epsilon \nabla_{\bf b} E \\
\end{align*}


In [ ]:
# Prevent log contents from becoming 0
def np_log(x):
    return np.log(np.clip(a=x, a_min=1e-10, a_max=1e+10))

In [ ]:
def train_or(x, y, eps=1.0):
    """
    :param x: np.ndarray, input data, (batch_size, Number of input dimensions)
    :param y: np.ndarray, teacher labels, (batch_size, Number of output dimensions)
    :param eps: float, learning rate
    """
    global W_or, b_or

    batch_size = x.shape[0]

    # prediction
    y_hat = sigmoid(np.matmul(x, W_or) + b_or)  # (batch_size, number of output dimensions)

    # Evaluation of objective function
    cost =  # WRITE ME

    delta = y_hat - y  # (batch_size, number of output dimensions)

    # Update parameters
    dW = np.matmul(x.T, delta) / batch_size  # (number of input dimensions, number of output dimensions)
    db = np.matmul(np.ones(shape=(batch_size,)), delta) / batch_size  # (number of dimensions of output,)
    W_or -= eps * dW
    b_or -= eps * db

    return cost

def valid_or(x, y):
    y_hat = sigmoid(np.matmul(x, W_or) + b_or)
    cost = (- y * np_log(y_hat) - (1 - y) * np_log(1 - y_hat)).mean()
    return cost, y_hat

### 2.4. 学習

---

### 2.4. Learning


In [ ]:
for epoch in range(1000):
    #  x_train_or, y_train_or = shuffle(x_train_or, y_train_or)

    cost = train_or(x_train_or, y_train_or)
    cost, y_pred = valid_or(x_valid_or, y_valid_or)

print(y_pred)

## 3.【課題 3】ソフトマックス回帰の実装と学習 (MNIST)

ソフトマックス回帰は，ロジスティック回帰を多クラス分類に拡張したものです．

---

## 3. [Assignment 3] Implementation and learning of softmax regression (MNIST)

Softmax regression is an extension of logistic regression to multiclass classification.


### 3.1. ソフトマックス関数

ソフトマックス関数は分子の全クラスにわたる和が分母になるので，モデルの出力を確率分布（多項分布）に直すものと考えることができます．

$$
\mathrm{softmax}({\bf x})_k
= \frac{\exp({\bf x}_k)}{\sum^K_{k'=1} \exp({\bf x}_{k'})}
= \frac{\exp({\bf x}_k - m)}{\sum^K_{k'=1} \exp({\bf x}_{k'} - m)} \hspace{10mm} \text{for} \, k=1,\ldots, K
$$

ソフトマックス関数は入力に対して定数を足し引きしても結果が変わりません．上式では，定数 $m = \max^K_{i=1} x_i$ を入力から引くことで，指数を取った時に値がオーバーフローしてしまうのを防ぐための実装を行っています．

---

### 3.1. Softmax function

Since the denominator of the softmax function is the sum over all classes of numerators, it can be thought of as converting the output of the model into a probability distribution (multinomial distribution).

$$
\mathrm{softmax}({\bf x})_k
= \frac{\exp({\bf x}_k)}{\sum^K_{k'=1} \exp({\bf x}_{k'})}
= \frac{\exp({\bf x}_k - m)}{\sum^K_{k'=1} \exp({\bf x}_{k'} - m)} \hspace{10mm} \text{for} \, k=1,\ldots, K
$$

The softmax function does not change the result even if a constant is added or subtracted from the input. In the above formula, the constant $m = \max^K_{i=1} x_i$ is subtracted from the input to prevent the value from overflowing when the exponent is taken.


In [ ]:
def softmax(x, axis=1):
    x -=  # WRITE ME # Prevent exp overflow
    x_exp =  # WRITE ME

    return  # WRITE ME


x = np.random.uniform(-5, 5, (1, 10))
y = softmax(x.copy())
fig, axs = plt.subplots(ncols=2, figsize=(8, 3))
for i, (d, title) in enumerate(zip([x, y], ['before', 'after'])):
    axs[i].bar(np.arange(10), d.squeeze())
    axs[i].set_title(f"{title} softmax")
    axs[i].set_xticks(np.arange(10))
    axs[i].set_xlabel("class")

$K=2$（2クラス）の際のソフトマックス関数を可視化してみます．

---

Let's visualize the softmax function when $K=2$ (2 classes).


In [ ]:
x = np.stack(np.meshgrid(
    np.arange(-5, 5, 0.1),
    np.arange(-5, 5, 0.1)
))
y = softmax(x.copy(), axis=0)

ax = plt.axes(projection='3d')
ax.view_init(elev=30, azim=45)
ax.plot_surface(x[0], x[1], y[0], cmap='Reds')
ax.plot_surface(x[0], x[1], y[1], cmap='Blues')
ax.set_xlabel("x_1")
ax.set_ylabel("x_2")

### 3.2. データセットの設定と重みの定義

次にデータセットを作成します．ここでは画像の多クラス分類タスクとして知られるMNISTデータセットを用います．MNISTデータセットは0-9の手書き文字の画像入力とし，画像にどの数字が書かれているかを出力する10クラス分類のタスクです．入力データは以下のようなグレースケールの画像が入力されます．

---

### 3.2. Dataset settings and weight definitions

Next, create a dataset. Here, we will use the MNIST dataset, which is known as a multi-class classification task for images. The MNIST dataset is a 10-class classification task that takes an image input of handwritten characters 0-9 and outputs which number is written in the image. The input data is a grayscale image like the one below.


In [ ]:
(x_mnist_1, y_mnist_1), (x_mnist_2, y_mnist_2) = mnist.load_data()

x_mnist = np.r_[x_mnist_1, x_mnist_2]
y_mnist = np.r_[y_mnist_1, y_mnist_2]

fig = plt.figure(figsize=(10, 10))

for i in range(20):
    x = x_mnist[i]
    ax = fig.add_subplot(10, 10, i+1, xticks=[], yticks=[])
    ax.imshow(x, 'gray')
fig.tight_layout(rect=[0,0,1,0.96])
plt.show()

In [ ]:
x_mnist = x_mnist.astype('float32') / 255.  # Pixel value scaling: 0 - 255 (integer) → 0.0 - 1.0 (float)
y_mnist = np.eye(N=10)[y_mnist.astype('int32').flatten()]  # Convert integer labels (0–9) to one-hot vector

x_mnist=x_mnist.reshape(x_mnist.shape[0],-1)  # Pixel flattening: 28 * 28 (2D image) → 784 (1D image)

x_train_mnist, x_test_mnist, y_train_mnist, y_test_mnist = train_test_split(x_mnist, y_mnist, test_size=10000)
x_train_mnist, x_valid_mnist, y_train_mnist, y_valid_mnist = train_test_split(x_train_mnist, y_train_mnist, test_size=10000)

In [ ]:
W_mnist = np.random.uniform(low=-0.08, high=0.08, size=(784, 10)).astype('float32')  # Weight: (784, 10)
b_mnist = np.zeros(shape=(10,)).astype('float32')

### 3.3. train関数とvalid関数

---

### 3.3. train and valid functions


**目的関数（Objective function）**

多クラス交差エントロピー誤差関数（Multiclass cross entropy error function）
$$ E ({\bf x}, {\bf y}; {\bf W}, {\bf b} ) =  -\frac{1}{N}\sum^N_{i=1} \sum^K_{k=1} {\bf y}_{i, k} \log {\bf \hat{y}}_{i, k} ({\bf x}_i; {\bf W}, {\bf b}) $$

**モデルの推論（Model Inference）**
$$
    {\bf \hat{y}}_i = \mathrm{softmax}({\bf W}^T{\bf x}_i + {\bf b})
$$

**モデルの学習（Training the model）**
\begin{align*}
    \delta_i &= {\bf \hat{y}}_i - {\bf y}_i \\
    \nabla_{\bf W} E &= \frac{1}{N}\sum^N_{i=1}x_i {\bf \delta}^{\mathrm{T}}_i \\
    \nabla_{\bf b} E &= \frac{1}{N}\sum^N_{i=1}\delta_i  \\
    {\bf W} &\leftarrow {\bf W} - \epsilon \nabla_{\bf W} E \\
    {\bf b} &\leftarrow {\bf b} - \epsilon \nabla_{\bf b} E \\
\end{align*}


In [ ]:
def train_mnist(x, y, eps=1.0):
    """
    :param x: np.ndarray, input data, (batch_size, Number of input dimensions)
    :param y: np.ndarray, teacher labels, (batch_size, Number of output dimensions)
    :param eps: float, learning rate
    """
    global W_mnist, b_mnist

    batch_size = x.shape[0]

    # prediction
    y_hat = softmax(np.matmul(x, W_mnist) + b_mnist)  # (batch_size, number of output dimensions)

    # Evaluation of objective function
    cost = (- y * np_log(y_hat)).sum(axis=1).mean()
    delta = y_hat - y  # (batch_size, number of output dimensions)

    # Update parameters
    dW = np.matmul(x.T, delta) / batch_size  # (number of input dimensions, number of output dimensions)
    db = np.matmul(np.ones(shape=(batch_size,)), delta) / batch_size  # (number of dimensions of output,)
    W_mnist -= eps * dW
    b_mnist -= eps * db

    return cost

def valid_mnist(x, y):
    y_hat = softmax(np.matmul(x, W_mnist) + b_mnist)
    cost = (- y * np_log(y_hat)).sum(axis=1).mean()

    return cost, y_hat

### 3.4. 学習

---

### 3.4. Learning


In [ ]:
for epoch in range(100):
    cost = train_mnist(x_train_mnist, y_train_mnist)

    cost, y_pred = valid_mnist(x_valid_mnist, y_valid_mnist)
    if epoch % 10 == 9 or epoch == 0:
        print('EPOCH: {}, Valid Cost: {:.3f}, Valid Accuracy: {:.3f}'.format(
            epoch + 1,
            cost,
            accuracy_score(y_valid_mnist.argmax(axis=1), y_pred.argmax(axis=1))
        ))